<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/kol_network_stance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Install — jalankan sekali di awal
import os, shutil, sys, importlib.util
import torch
from google.colab import userdata

try:
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception as e:
    raise RuntimeError(
        f"Secret GH_TOKEN belum bisa dibaca ({type(e).__name__}). Cara memperbaiki:\n"
        "  1. Klik ikon 🔑 (Secrets) di sidebar kiri Colab\n"
        "  2. Klik 'Add new secret'. Name: GH_TOKEN, Value: token GitHub kamu\n"
        "     (token harus punya akses ke repo azmkto/GREAT-Tools; minta ke admin kalau belum punya)\n"
        "  3. Nyalakan toggle 'Notebook access' di baris GH_TOKEN\n"
        "  4. Jalankan ulang cell ini"
    ) from None

REPO_DIR = "/content/GREAT-Tools"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q https://{GH_TOKEN}@github.com/azmkto/GREAT-Tools.git {REPO_DIR}
if not os.path.isdir(f"{REPO_DIR}/great"):
    raise RuntimeError("Gagal download repo GREAT-Tools. Token GH_TOKEN salah, kedaluwarsa, "
                       "atau tidak punya akses ke repo azmkto/GREAT-Tools. Perbarui secret-nya lalu jalankan ulang.")

# torch, pandas, tqdm, huggingface_hub bawaan Colab dipakai apa adanya (torch-nya sudah versi GPU).
# Meng-install ulang paket yang sudah ter-import = wajib restart, jadi itu yang dihindari.
PACKAGES = ["emoji", "openpyxl", "ftfy", "networkx"]
already_loaded = [p for p in [*PACKAGES, "llama_cpp"] if p in sys.modules]
PIP_ARGS = " ".join(PACKAGES)
!pip install -q -U {PIP_ARGS}

# LLM lokal (Step 6): llama-cpp-python versi prebuilt. abetlen baru menyediakan wheel CUDA 12 (cu121..cu124).
# Colab sekarang memakai CUDA 13: driver-nya tetap bisa menjalankan program CUDA 12, yang kurang hanya
# library runtime CUDA 12 -> dipasang lewat pip, lalu dimuat di Step 6A sebelum llama_cpp di-import.
HAS_GPU = torch.cuda.is_available()
cuda_major, cuda_minor = (torch.version.cuda or "0.0").split(".")[:2]
if HAS_GPU and cuda_major not in ("12", "13"):
    raise RuntimeError(f"CUDA {torch.version.cuda}: belum didukung. Ganti runtime ke T4 GPU.")
if not HAS_GPU:
    CU_TAG = "cpu"
elif cuda_major == "13":
    CU_TAG = "cu124"
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
else:
    CU_TAG = f"cu12{min(int(cuda_minor), 4)}"
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{CU_TAG}
if not HAS_GPU:
    print("⚠️ Tidak ada GPU: LLM lokal akan SANGAT lambat. Runtime -> Change runtime type -> T4 GPU.")

missing = [m for m in ["torch", "tqdm", "huggingface_hub", "llama_cpp", *PACKAGES] if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Paket gagal terpasang: {missing}. Scroll ke atas, cari error pip-nya.")

if already_loaded:
    # hanya terjadi kalau cell ini dijalankan dua kali di session yang sama
    print(f"[PERLU RESTART] {already_loaded} sudah ter-import sebelum di-update. "
          "Klik Runtime -> Restart session, lalu lanjut dari Cell ② (tidak perlu jalankan cell ini lagi).")
else:
    print(f"[OK] Semua paket siap (torch CUDA {torch.version.cuda}, llama-cpp {CU_TAG}), tidak perlu restart. "
          "Lanjut: isi form di Cell ②, klik Cell ②, lalu Runtime -> Run cell and below.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 560.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
[OK] Semua paket siap (torch CUDA 13.0, llama-cpp cu124), tidak perlu restart. Lanjut: isi form di Cell ②, klik Cell ②, lalu Runtime -> Run cell and below.


## Import Library

In [ ]:
import json, os, random, re, sys, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import emoji
import networkx as nx
import pandas as pd
from google.colab import ai, drive
from IPython.display import display
from tqdm.auto import tqdm

# restart runtime menghapus sys.path & variabel, jadi repo didaftarkan ulang di sini
REPO_DIR = "/content/GREAT-Tools"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from great import EXPORT_COLUMNS, clean_for_bert, is_media_account

tqdm.pandas()

print("great loaded from:", sys.modules["great"].__file__)
print("Commit terakhir  :", os.popen(f"git -C {REPO_DIR} rev-parse --short HEAD").read().strip())
print("Model Colab AI   :", ai.list_models())

great loaded from: /content/GREAT-Tools/great/__init__.py
Commit terakhir  : 99f3d60
Model Colab AI   : ['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-3.5-flash']


# Colab AI

In [ ]:
AI_MODEL = "google/gemini-2.5-flash"   # pilih dari ai.list_models()
SLEEP_BETWEEN_CALLS = 4
api_call_count = 0


def is_quota_error(e):
    msg = str(e).lower()
    return "429" in msg or "resource_exhausted" in msg or "quota" in msg


def parse_json_text(text):
    """JSON pertama di jawaban model (Colab AI tidak punya mode JSON)."""
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    if not starts:
        raise ValueError("tidak ada JSON di jawaban model")
    return json.JSONDecoder().raw_decode(text[min(starts):])[0]


def ask_json(prompt, parse=lambda x: x, retry=4):
    """Colab AI -> JSON -> parse(). Kuota habis langsung di-raise; None kalau semua retry gagal."""
    global api_call_count
    for attempt in range(retry):
        try:
            api_call_count += 1
            return parse(parse_json_text(ai.generate_text(prompt, model_name=AI_MODEL)))
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = (20 if "503" in str(e) or "UNAVAILABLE" in str(e) else 5) * (attempt + 1)
            print(f"[WARN] AI gagal ({attempt + 1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return None


def order_by_index(parsed, n):
    """List objek ber-field 'index' (1..n) -> list urut. Error kalau ada index yang hilang."""
    by_idx = {int(it["index"]): it for it in parsed if isinstance(it, dict) and "index" in it}
    if set(by_idx) != set(range(1, n + 1)):
        raise ValueError(f"Index hasil tidak lengkap: dapat {sorted(by_idx)}, harus 1..{n}")
    return [by_idx[i] for i in range(1, n + 1)]

In [ ]:
PROJECT = "prabowo_vladivostok"
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
STANCE_TARGET = "Presiden Prabowo dan pemerintah"
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
REGENERATE_TOPIC_CONFIG = False

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC = TOPIC.strip()
STANCE_TARGET = STANCE_TARGET.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]

problems = [msg for ok, msg in [(PROJECT, "PROJECT masih kosong"),
                                (TOPIC, "TOPIC masih kosong"),
                                (STANCE_TARGET, "STANCE_TARGET masih kosong"),
                                (SOURCES, "SOURCES_TEXT masih kosong (isi link Google Sheets)")] if not ok]
if problems:
    raise ValueError("Form belum lengkap: " + "; ".join(problems))
print(f"[OK] PROJECT = {PROJECT}\n     TOPIC   = {TOPIC}\n     STANCE  = sikap terhadap {STANCE_TARGET}\n"
      f"     {len(SOURCES)} file sumber")

[OK] PROJECT = prabowo_vladivostok
     TOPIC   = Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin
     STANCE  = sikap terhadap Presiden Prabowo dan pemerintah
     1 file sumber


# Konfigurasi
Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [ ]:
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive"

COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_MEDIA = "Media"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]   # Location tidak dipakai di sini

# LLM lokal untuk sentimen + stance + summary (Step 6-7), sama dengan GREAT-Tools/Notebook/topic_modelling.ipynb
LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192

# Skor volume (Step 5), jumlahnya 1.0
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15
# Skor jaringan (Step 5B), jumlahnya 1.0
NET_WEIGHT_PAGERANK = 0.7   # pengaruh yang mengalir: di-retweet/mention oleh akun yang juga berpengaruh
NET_WEIGHT_EIGEN = 0.3      # Eigenvector: terhubung dengan akun-akun yang sama-sama terhubung kuat
# Skor KOL final (Step 5B) = volume + jaringan, jumlahnya 1.0
KOL_WEIGHT_VOLUME = 0.5    # jumlah post asli + engagement + followers
KOL_WEIGHT_NETWORK = 0.5   # PageRank + Eigenvector

BATCH_SIZE_CLASSIFY = 40     # cuitan per panggilan Colab AI saat klasifikasi tema (Step 4)
BATCH_SIZE_VALIDATE = 5      # KOL per panggilan Colab AI saat validasi sentimen & stance (Step 6C)
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

RANKING_MIN_CONFIDENCE = 0.6      # tema di bawah ini tidak ikut ranking & jaringan
DEDUP_PER_AUTHOR = False          # True = post kembar (copas) dari author berbeda tetap dihitung

# Merge akun: satu akun/outlet bisa muncul dengan beberapa ejaan -> satu key (dari author_network_extractor)
ALIASES = {}   # key -> key, untuk merge yang terlewat aturan. Contoh: {"republikaonline": "republikacoid"}
TWO_PART_SLD = {"co", "or", "ac", "go", "web", "my", "sch", "net", "com"}


def canonical(name):
    """Key merge akun: '@KOMPASTVCOM', 'www.kompastv.com' -> 'kompastvcom'; 'prabowo.subianto' = '@Prabowo_Subianto'.
    Author kosong -> None (bukan 'nan'), supaya post tanpa Author tidak digabung jadi satu akun palsu."""
    if pd.isna(name) or not str(name).strip():
        return None
    s = re.sub(r"^(https?://)?(www\.)?@?", "", str(name).strip().lower()).split("/")[0]
    parts = s.split(".")
    if len(parts) > 2 and re.fullmatch(r"[a-z]{2,3}", parts[-1]):   # domain, bukan handle IG bertitik
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r"[^a-z0-9]", "", "".join(parts))
    return ALIASES.get(key, key)


assert canonical("@KOMPASTVCOM") == canonical("www.kompastv.com") == "kompastvcom"
assert canonical("nasional.kompas.com") == canonical("@kompascom") == "kompascom"
assert canonical("https://www.republika.co.id/berita/x") == "republikacoid"
assert canonical("prabowo.subianto") == canonical("@Prabowo_Subianto") == "prabowosubianto"
assert canonical(float("nan")) is None and canonical("  ") is None

TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
CHECKPOINT_PATH = f"{DRIVE_DIR}/checkpoint_4_{PROJECT}.pkl"
# Hasil (KOL, nodes, edges, sentimen, stance, ringkasan) TIDAK disimpan ke file, cukup ditampilkan penuh di notebook.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Mounted at /content/drive


# Konfigurasi topik dari Colab AI

In [ ]:
TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia yang menyiapkan konfigurasi klasifikasi otomatis.
Topik yang dipantau: "{TOPIC}".
Tanggal hari ini: {date.today()}. Jangan menulis tahun di keyword kecuali yakin tahunnya benar.

Konfigurasi ini dipakai oleh pipeline berikut, jadi ikuti aturan tiap field dengan ketat:
1. Cuitan yang TIDAK mengandung satu pun `keywords` langsung dibuang sebagai "lainnya" tanpa dibaca AI.
   Keyword yang terlewat = cuitan relevan hilang diam-diam. Keyword terlalu umum = ribuan cuitan
   tidak relevan ikut diproses (lama & boros kuota).
2. Cuitan yang lolos dibaca AI dan diberi satu tema berdasarkan deskripsi di `themes`.
3. Akun di `media_accounts` ditandai sebagai akun media.

PENTING: data mentah biasanya hasil monitoring yang LEBIH LUAS dari topik ini, misalnya SEMUA cuitan
yang menyebut tokoh utamanya (termasuk isu lain yang tidak berhubungan). Karena itu nama tokoh utama
saja BUKAN penanda topik ini.

=== ATURAN PER FIELD ===

themes (objek: nama_tema -> deskripsi)
- 2-4 tema yang membagi percakapan tentang topik ini secara jelas. Tiap cuitan relevan harus
  masuk tepat satu tema. Pilih pembagian berdasarkan sudut pandang percakapan (misal: agenda/acara,
  isu kebijakan/kerja sama, reaksi politik/geopolitik), bukan berdasarkan tokoh.
- Nama: snake_case, singkat, bahasa Indonesia (contoh: kerja_sama_ekonomi).
- Deskripsi satu string dengan format:
  "Masuk: <apa yang dibahas, 1 kalimat>. Contoh frasa: <8-15 frasa khas cuitan, termasuk bahasa
  gaul, singkatan, julukan>. Bukan: <yang sering tertukar dengan tema lain dan masuk ke tema mana>."
- Jangan buat tema "lainnya", "umum", atau "campuran". Itu sudah disediakan sistem.

keywords (list string)
- 20-60 kata/frasa, huruf kecil semua, dicocokkan sebagai frasa PERSIS dan KATA UTUH.
  Contoh: "putin" TIDAK cocok dengan "putinisme"; "prabowo putin" TIDAK cocok dengan "prabowo bertemu putin".
  Jadi utamakan kata tunggal yang khas, bukan frasa panjang.
- Isi dengan penanda yang KHAS topik ini: nama pihak lain, tempat, acara, dan instansi yang hampir
  hanya muncul saat membicarakan topik ini (ejaan Indonesia dan Inggris, singkatan, salah ketik umum,
  bentuk hashtag digabung tanpa #).
- DILARANG: nama tokoh utama sendirian (misal "prabowo", "presiden prabowo", julukannya), dan kata umum
  yang juga dipakai di isu lain (misal "investasi", "pertahanan", "kesepakatan", "perdamaian",
  "kunjungan kerja", "kebijakan luar negeri", "ketahanan pangan").

media_accounts (list string)
- Username akun berita/media dan kanal resmi institusi yang khusus terkait topik ini
  (contoh: kantor berita negara asing, akun humas lembaga). Huruf kecil, tanpa @.
- JANGAN masukkan akun pribadi tokoh politik, buzzer, atau influencer: mereka justru
  harus ikut dianalisis sebagai author.
- Kalau tidak yakin username-nya ada, jangan dimasukkan. List kosong boleh.

=== CEK SEBELUM MENJAWAB ===
- Tidak ada keyword berupa nama tokoh utama sendirian atau kata umum.
- Setiap tema punya bagian "Bukan:" yang menyebut batas dengan tema lain.
- Semua keywords huruf kecil.

=== FORMAT JAWABAN ===
Jawab HANYA satu objek JSON, tanpa markdown, tanpa ```, tanpa teks sebelum atau sesudahnya:
{{"themes": {{"nama_tema": "Masuk: ... Contoh frasa: ... Bukan: ..."}},
  "keywords": ["...", "..."],
  "media_accounts": ["..."]}}"""

# config lama hanya dipakai kalau disusun untuk TOPIC yang sama persis
topic_config = None
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        cached = json.load(f)
    if cached.get("_topic") == TOPIC:
        topic_config = cached
        print(f"[INFO] Konfigurasi topik dibaca dari {TOPIC_CONFIG_PATH}")
    else:
        print("[INFO] TOPIC berbeda dari konfigurasi yang tersimpan -> AI menyusun ulang")

if topic_config is None:
    topic_config = ask_json(TOPIC_CONFIG_PROMPT)
    if not topic_config or not topic_config.get("themes"):
        raise RuntimeError("Colab AI gagal menyusun konfigurasi topik (themes kosong). Jalankan ulang cell ini.")
    topic_config["_topic"] = TOPIC
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)
    print(f"[INFO] Konfigurasi topik dari AI disimpan ke {TOPIC_CONFIG_PATH}")

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = [t for t in THEMES if t != "lainnya"]
KW_RELEVAN = [k.lower() for k in topic_config.get("keywords", [])]
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config.get("media_accounts", [])}

# Baris tanpa satu pun keyword langsung "lainnya" (hemat kuota AI). Tanpa keyword = semua baris ke AI.
RE_RELEVAN = re.compile(
    r"(?<!\w)(?:" + "|".join(map(re.escape, sorted(set(KW_RELEVAN), key=len, reverse=True))) + r")(?!\w)"
) if KW_RELEVAN else None


print(json.dumps(topic_config, ensure_ascii=False, indent=2))

[INFO] Konfigurasi topik dari AI disimpan ke /content/drive/MyDrive/topic_config_prabowo_vladivostok.json
{
  "themes": {
    "agenda_pertemuan": "Masuk: Pembahasan detail terkait jadwal, lokasi, susunan acara, dan siapa saja yang terlibat dalam kunjungan Presiden Prabowo ke Vladivostok serta pertemuannya dengan Vladimir Putin. Contoh frasa: tiba vladivostok, agenda prabowo putin, pertemuan bilateral, kunjungan kenegaraan, delegasi indonesia, jadwal presiden, sambutan hangat, jam berapa. Bukan: Hasil perjanjian atau kerja sama (masuk kerja_sama_bilateral), atau tanggapan/analisis politis (masuk reaksi_geopolitik).",
    "kerja_sama_bilateral": "Masuk: Diskusi mengenai potensi kesepakatan, MoU, atau area kerja sama spesifik antara Indonesia dan Rusia, termasuk sektor pertahanan, energi, ekonomi, pangan, dan teknologi. Contoh frasa: kesepakatan pertahanan, kerjasama energi, investasi rusia, MoU indonesia rusia, pangan, alutsista, perdagangan bilateral, ekonomi strategis, pasokan minyak, 

# Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [ ]:
def xlsx_url(link):
    """Link Google Sheets/Drive -> url download .xlsx. Selain itu (path lokal) dipakai apa adanya."""
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastikan akses file 'Anyone with the link'.") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # KeyError menyebut kolom yang hilang


# Duplikat post lintas file dibuang di Step 3 (dedup_key).
df_raw = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

Data berhasil dimuat: (66891, 13)


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,source_file
0,1,mention,"Netflix Rilis Skenario Sang Jenderal, Film Dokumenter tentang Kasus Ferdy Sambo","rmol news logo Film dokumenter Skenario Sang Jenderal tayang di Netflix dan mengangkat kembali kasus pembunuhan Brigadir Polisi Nofriansyah Yosua Hutabarat atau Brigadir J yang melibatkan mantan petinggi Polri, Ferdy Sambo. Film dokumenter Skenario Sang Jenderal diproduksi oleh Bleach House Pictures bekerja sama dengan Tempo Media.\n\nDokumenter tersebut mengulas rangkaian peristiwa yang terjadi sejak kematian Brigadir J pada Juli 2022 hingga proses hukum yang menyusul setelahnya. Disutradarai Rob Sixsmith, Skenario Sang Jenderal memiliki durasi 1 jam 29 menit.\n\nFilm ini menampilkan sejumlah rekaman dan gambar terkait kasus tersebut, termasuk kondisi tempat kejadian perkara, proses persidangan, serta keluarga Brigadir J.\n\nFilm dokumenter Skenario Sang Jenderal dibuka dengan narasi mengenai berbagai informasi yang beredar setelah Brigadir J ditemukan tewas. Selain rekaman dari lokasi kejadian, dokumenter ini menghadirkan proses persidangan sejumlah terdakwa.\n\nKesaksian dan keterangan dari sejumlah narasumber juga digunakan untuk membangun kronologi kasus yang sempat menjadi perhatian luas di Indonesia. Film tersebut turut memperlihatkan duka keluarga Brigadir J setelah kematian Yosua.\n\nYosua ditemukan tewas di rumah dinas Ferdy Sambo di kompleks Polri, Duren Tiga, Jakarta Selatan. Pada awal pengungkapan kasus, muncul narasi bahwa Brigadir J tewas dalam baku tembak dengan Bharada Richard Eliezer atau Bharada E.\n\nPeristiwa tersebut dikaitkan dengan dugaan pelecehan seksual terhadap Putri Candrawathi, istri Ferdy Sambo. Seiring proses penyelidikan, narasi awal tersebut kemudian berubah.\n\nKapolri Jenderal Listyo Sigit Prabowo menyatakan bahwa tidak terjadi baku tembak seperti yang sebelumnya disampaikan. Penyidikan selanjutnya mengungkap keterlibatan Ferdy Sambo dalam pembunuhan Brigadir J.\n\nSambo dinyatakan terbukti memerintahkan Bharada E menembak Yosua dan melakukan rekayasa tempat kejadian perkara untuk membangun narasi seolah-olah terjadi baku tembak. Kasus tersebut juga berkembang ke dugaan perusakan dan penghilangan barang bukti, termasuk rekaman kamera CCTV di sekitar lokasi kejadian.\n\nPerkara kemudian mencakup pembunuhan berencana serta obstruction of justice atau tindakan menghalangi proses hukum.\n\nLima orang menjadi terdakwa dalam perkara pembunuhan berencana Brigadir J, yakni Ferdy Sambo, Putri Candrawathi, Richard Eliezer, Ricky Rizal, dan Kuat Ma&#39;ruf. Pengadilan Negeri Jakarta Selatan pada Februari 2023 menjatuhkan hukuman mati kepada Ferdy Sambo.\n\nPutusan tersebut kemudian berubah menjadi hukuman penjara seumur hidup pada tingkat banding. Putri Candrawathi dijatuhi hukuman 20 tahun penjara.\n\nSementara itu, Richard Eliezer atau Bharada E mendapatkan hukuman lebih ringan setelah statusnya sebagai justice collaborator menjadi salah satu pertimbangan dalam proses hukum.\n\nKasus pembunuhan Brigadir J menjadi salah satu perkara yang mendapat perhatian nasional karena melibatkan pejabat tinggi kepolisian. Perkara tersebut juga memunculkan pembahasan mengenai penyalahgunaan kekuasaan, proses penyidikan, serta penanganan barang bukti dalam institusi penegak hukum.&quot;https://dashboard.rmol.id/assets/images/logo/10441704062019_akhir.png&quot;&gt;\n\n",2026-09-22 15:45:00,https://rmol.id/hiburan/read/2026/09/22/722557/netflix-rilis-skenario-sang-jenderal-film-dokumenter-tentang-kasus-ferdy-sambo,News,Negative,rmol.id,0,0,0,file_1
1,2,rt,NaN,"RT Negara hancur, Aturan baru OJK: Danantara, BI, kementerian keuangan, boleh punya saham BEI 😂 Heiii antek-antek asinggg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo 😂",2026-09-22 15:32:35,https://twitter.com/web/statuses/2102315108575965632,Twitter,Negative,@iwantjiminp,5030,0,0,file_1
2,

# Step 2 - Filter akun media (satu-satunya definisi + eksekusi)


In [ ]:
# True = akun media (outlet berita, kanal TV/YouTube) ikut dianalisis dan bisa jadi KOL; tetap diberi tanda is_media.
# False = akun media dibuang di sini, seperti versi sebelumnya (hanya akun non-media).
INCLUDE_MEDIA = False

NUM_COLS = [COL_FOLLOWERS, COL_RETWEETED, COL_FAVOURITED]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

# Author di YouTube/Facebook berupa nama tampilan (ada spasi), tidak dikenali filter username di library.
MEDIA_NAME_WORDS = {"tv", "berita", "news", "info", "terkini", "update", "channel", "official",
                    "media", "portal", "redaksi", "production", "productions"}


def is_media_name(author):
    """'TV Rakyat | Seputar Info Terkini', 'KOMPASTV JAWA BARAT' -> True. Username tanpa spasi tidak dicek di sini."""
    author = str(author).strip()
    words = re.findall(r"[a-z]+", author.lower())
    return " " in author and any(w in MEDIA_NAME_WORDS or w.endswith("tv") for w in words)


assert is_media_name("TV Rakyat | Seputar Info Terkini") and is_media_name("KOMPASTV JAWA BARAT")
assert not is_media_name("Ahmad Dedi") and not is_media_name("@newsfan")

by_account = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                        for a, p in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])], index=df_raw.index)
by_name = df_raw[COL_AUTHOR_ID].map(is_media_name)
is_media = by_account | by_name

# merged media: outlet yang ketahuan media lewat SATU ejaan (mis. www.kompastv.com di News) ikut ditandai media
# di ejaan lainnya (@KOMPASTVCOM di Twitter). MEDIA_IDS dipakai lagi di Step 5B untuk node jaringan;
# kolom author_key (key canonical) dipakai lagi di Step 5 dan 5B.
df_raw["author_key"] = df_raw[COL_AUTHOR_ID].map(canonical)
MEDIA_IDS = set(df_raw.loc[is_media, "author_key"].dropna()) | {canonical(a) for a in EXTRA_MEDIA_ACCOUNTS}
by_merge = df_raw["author_key"].isin(MEDIA_IDS) & ~is_media
is_media |= by_merge
df_raw["is_media"] = is_media

print(f"[INFO] {is_media.sum()} baris media terdeteksi ({(by_name & ~by_account).sum()} dari nama kanal, "
      f"{by_merge.sum()} dari ejaan lain outlet yang sama), {(~is_media).sum()} baris non-media")
print("[INFO] Contoh kanal media dari nama:", df_raw.loc[by_name & ~by_account, COL_AUTHOR_ID].unique()[:15].tolist())
print("[INFO] Contoh ejaan media dari merge:", df_raw.loc[by_merge, COL_AUTHOR_ID].unique()[:15].tolist())
display(df_raw.assign(is_media=is_media)[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID).head(20))

if INCLUDE_MEDIA:
    print(f"[INFO] INCLUDE_MEDIA = True: {is_media.sum()} baris media IKUT dianalisis. "
          "Artikel berita panjang & banyak -> Step 4D lebih lama dan lebih boros kuota Colab AI.")
else:
    df_raw = df_raw[~is_media].reset_index(drop=True)
    print("[INFO] INCLUDE_MEDIA = False: baris media dibuang.")
print(f"[INFO] {df_raw['author_key'].isna().sum()} baris tanpa Author: tetap dipakai untuk tema, tapi tidak ikut ranking KOL")

[INFO] 29558 baris media terdeteksi (790 dari nama kanal, 77 dari ejaan lain outlet yang sama), 37333 baris non-media
[INFO] Contoh kanal media dari nama: ['channel Irfanmolay ', 'AGH Update', 'Viral & Update ', 'KOMPASTV BENGKULU', 'Edi berita', 'KASIH BERITA', 'KOMPASTV JAWA BARAT', 'YTNDR INFO', 'Roll Channel Solo', 'Nanang tv mesuji', 'Gus Azka Channel ', 'Celah Info', 'Singkat Berita', 'W21 Channel', 'Rahman Channel']
[INFO] Contoh ejaan media dari merge: ['@pantaucom_', 'Akuratco', '@jpnncom', 'priangan com', '@historia_id', '@geloraco', 'SUARANTBcom', '@nusabalicom', 'merdekacom', 'Blok-acom', 'idxchannel', 'IndonesiaGOID', 'jpnncom', 'SUARA PUBLIK ID ', 'Ponorogo Upda Te']


,Author,Media,is_media
0,rmol.id,News,True
1,@iwantjiminp,Twitter,False
2,www.kompas.com,News,True
4,www.cnnindonesia.com,News,True
5,@wanzhouvers,Twitter,False
6,economy.okezone.com,News,True
8,index.okezone.com,News,True
10,@moodmasyarakat,Twitter,False
11,wartaekonomi.co.id,News,True
16,nasional.tvrinews.com,News,True


[INFO] INCLUDE_MEDIA = False: baris media dibuang.
[INFO] 11477 baris tanpa Author: tetap dipakai untuk tema, tapi tidak ikut ranking KOL


# Step 3 — Cleaning text (jangan diubah)

In [ ]:
# Export ini menulis retweet sebagai "RT <teks>" (sering TANPA @user) dan balasan sebagai "... [RE username]".
RE_RT_PREFIX = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")      # ponytail: "RT" huruf besar di awal = retweet
RE_RT = re.compile(r"(?<!\w)RT\s*@(\w+)", re.IGNORECASE)        # akun yang di-retweet, kalau tertulis
RE_REPLY = re.compile(r"\[RE\s+@?(\w+)\]", re.IGNORECASE)        # akun yang dibalas
RE_ELONGATION = re.compile(r"(.)\1{2,}")
EMOJI_LANG = "id" if "id" in emoji.LANGUAGES else "en"


def strip_noise(text):
    """Buang prefix RT & penanda [RE user], emoji -> kata, 'mantaaap' -> 'mantaap'."""
    text = RE_REPLY.sub(" ", RE_RT_PREFIX.sub("", text))
    text = emoji.demojize(text, language=EMOJI_LANG).replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


assert strip_noise("RT @abc: halo [RE def]").strip() == "halo"
assert strip_noise("RT halo").strip() == "halo"


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


def dedup_key(texts):
    """Teks -> kunci pembanding duplikat (huruf kecil, tanpa tanda baca). Dipakai lagi di Step 5B."""
    return texts.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")


df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_raw[COL_HEADLINE], df_raw[COL_MENTIONS])]
df_raw["is_retweet"] = df_raw["text_raw_combined"].map(lambda t: bool(RE_RT_PREFIX.match(t)))
df_raw["text_clean"] = df_raw["text_raw_combined"].map(strip_noise).progress_apply(clean_for_bert)

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3].copy()
print(f"[INFO] Buang {len(df_raw) - len(df_clean)} baris kosong/terlalu pendek")

before = len(df_clean)
df_clean = (df_clean.assign(dedup_key=dedup_key(df_clean["text_clean"]))
            .sort_values("is_retweet", kind="stable")   # post asli didahulukan: yang tersimpan milik penulis aslinya
            .drop_duplicates(subset=["dedup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dedup_key"])
            .sort_index()
            .drop(columns="dedup_key").reset_index(drop=True))
print(f"[INFO] {df_raw['is_retweet'].sum()} retweet terdeteksi. Buang {before - len(df_clean)} duplikat "
      f"(per author: {DEDUP_PER_AUTHOR})")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning ({df_clean['is_retweet'].sum()} retweet tanpa post asli di data)")

display(df_clean[["text_raw_combined", "text_clean"]].head(5))

  0%|          | 0/37333 [00:00<?, ?it/s]

[INFO] Buang 1709 baris kosong/terlalu pendek
[INFO] 13816 retweet terdeteksi. Buang 14164 duplikat (per author: False)
[OK] 21460 baris tersisa setelah cleaning (936 retweet tanpa post asli di data)


,text_raw_combined,text_clean
0,"RT Negara hancur, Aturan baru OJK: Danantara, BI, kementerian keuangan, boleh punya saham BEI 😂 Heiii antek-antek asinggg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo 😂","Negara hancur, Aturan baru OJK Danantara, BI, kementerian keuangan, boleh punya saham BEI wajah gembira berurai air mata Heii antek-antek asingg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo wajah gembira berurai air mata"
1,"RT ""Prabowo NPD this NPD that.."" . . . .. tapi ntar milih KDM https://t.co/kijt0OfoiU","""Prabowo NPD this NPD that.."" . . . .. tapi nanti milih KDM //t.co/kijt0OfoiU"
2,Pakar Hukum UNPAD Romli Atmasasmita: Presiden Prabowo percaya kepada kinerja Kapolri Listyo Sigit Prabowo🫡 #MoodMasyarakat #Kapolri #Unpad #ListyoSigitPrabowo #FYP https://t.co/iBMKU6uKAN,Pakar Hukum UNPAD Romli Atmasasmita Presiden Prabowo percaya kepada kinerja Kapolri Listyo Sigit Prabowo wajah memberi hormat MoodMasyarakat Kapolri Unpad ListyoSigitPrabowo FYP //t.co/iBMKU6uKAN
3,"emang prabowo sialan, mbg sialan [RE kenhans03]","memang prabowo sialan, mbg sialan"
4,RT breaking news gibran lagi di gugat di mk penggugat minta diskualifikasi gibran dari cawapres soalnya dianggap enggak penuhi syarat pendidikan lalu minta mpr gelar sidang buat milih wapres baru dari dua calon usulan prabowo https://t.co/S8DNK7XE5k,breaking news gibran lagi di gugat di mk penggugat minta diskualifikasi gibran dari cawapres soalnya dianggap tidak penuhi syarat pendidikan lalu minta mpr gelar sidang buat milih wapres baru dari dua calon usulan prabowo //t.co/S8DNK7XE5k


# Step 4 — Klasifikasi tema (prefilter keyword + Colab AI).

In [ ]:
AI_WORKERS = 1       # request AI paralel. 2 sudah kena rate limit di Colab AI; naikkan hanya kalau kuotamu besar.
MAX_AI_ROWS = None   # None = semua baris. Angka = sampel acak (ranking Step 5 jadi dari sampel).
CHECKPOINT_EVERY = 10   # batch

THEME_LIST_STR = "\n".join(f"- {t}: {d}" for t, d in THEME_DESCRIPTIONS.items())


def build_theme_prompt(texts):
    joined = "\n".join(f'{i}. "{t}"' for i, t in enumerate(texts, 1))
    return f"""Kamu classifier tema untuk cuitan bahasa Indonesia tentang: {TOPIC}.

Daftar tema:
{THEME_LIST_STR}

Tentukan satu tema untuk setiap cuitan.
- Tema "lainnya" WAJIB dipakai kalau cuitan tidak membahas topik di atas secara langsung
  (misal hanya menyebut nama tokoh, negara, atau acara yang sama untuk peristiwa lain).

Cuitan:
{joined}

Jawab HANYA JSON array berisi tepat {len(texts)} objek dengan field:
index (nomor cuitan), primary_theme (salah satu nama tema di atas), confidence (0-1)."""


# sidik jari run ini: checkpoint dari topik/tema/data yang berbeda tidak boleh dilanjutkan
RUN_FINGERPRINT = {
    "topic": TOPIC,
    "themes": THEMES,
    "n_rows": len(df_clean),
    "data_hash": int(pd.util.hash_pandas_object(df_clean["text_clean"], index=False).sum()),
}


def save_checkpoint(next_b):
    pd.to_pickle({"fingerprint": RUN_FINGERPRINT, "df": df_clean, "review_idx": review_idx, "next_b": next_b},
                 CHECKPOINT_PATH)


def load_checkpoint():
    """Checkpoint milik run ini, atau None. Yang tidak bisa dibaca (beda versi pandas, file rusak)
    atau milik data/topik lain dihapus supaya run mulai dari awal, bukan crash."""
    if not os.path.exists(CHECKPOINT_PATH):
        return None
    try:
        ckpt = pd.read_pickle(CHECKPOINT_PATH)
    except Exception as e:
        print(f"[INFO] Checkpoint lama tidak bisa dibaca ({type(e).__name__}) -> dihapus, mulai dari awal")
        os.remove(CHECKPOINT_PATH)
        return None
    if ckpt.get("fingerprint") != RUN_FINGERPRINT:
        print("[INFO] Checkpoint lama berasal dari data/topik yang berbeda -> dihapus, mulai dari awal")
        os.remove(CHECKPOINT_PATH)
        return None
    return ckpt


def classify_batch(b):
    """Satu request AI (jalan di thread). Return (index df, hasil per cuitan atau None)."""
    idx_batch = review_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    texts = df_clean.loc[idx_batch, "text_clean"].tolist()
    # hasil dicocokkan lewat field 'index', BUKAN posisi
    return idx_batch, ask_json(build_theme_prompt(texts), parse=lambda p: order_by_index(p, len(texts)), retry=6)


ckpt = load_checkpoint()
if ckpt is not None:
    df_clean, review_idx, start_b = ckpt["df"], ckpt["review_idx"], ckpt["next_b"]
    print(f"[RESUME] Lanjut dari batch {start_b} (checkpoint {CHECKPOINT_PATH})")
else:
    df_clean[["theme", "theme_confidence"]] = None
    df_clean["from_prefilter"] = False

    # tahap 1: keyword prefilter -> tanpa satu pun keyword relevan langsung "lainnya"
    if RE_RELEVAN:
        text_lower = df_clean["text_clean"].str.lower()
        irrelevant = ~text_lower.str.contains(RE_RELEVAN)
        df_clean.loc[irrelevant, ["theme", "theme_confidence", "from_prefilter"]] = ["lainnya", 1.0, True]

        kw_hits = Counter(k for t in text_lower[~irrelevant] for k in set(RE_RELEVAN.findall(t)))
        print("[INFO] Keyword yang paling banyak meloloskan baris:", kw_hits.most_common(10))
        if (~irrelevant).mean() > 0.5:
            print(f"[WARN] {(~irrelevant).mean():.0%} baris lolos prefilter: ada keyword yang terlalu umum (lihat di atas). "
                  "Proses jadi lama & boros kuota. Centang REGENERATE_TOPIC_CONFIG di Cell ② lalu jalankan ulang dari sana.")
    review_idx = df_clean.index[df_clean["theme"].isna()].tolist()
    print(f"[INFO] Prefilter: {df_clean['from_prefilter'].sum()} baris 'lainnya', {len(review_idx)} lanjut ke AI")
    if MAX_AI_ROWS and len(review_idx) > MAX_AI_ROWS:
        review_idx = sorted(random.sample(review_idx, MAX_AI_ROWS))
        print(f"[WARN] Sampling: hanya {MAX_AI_ROWS} baris diproses, sisanya tanpa tema (ranking dari sampel)")
    start_b = 0

# tahap 2: AI tentukan tema, AI_WORKERS batch sekaligus
n_batches = -(-len(review_idx) // BATCH_SIZE_CLASSIFY)
print(f"[INFO] Sisa {n_batches - start_b} batch, {AI_WORKERS} paralel")
pbar = tqdm(total=n_batches, initial=start_b, desc="Klasifikasi tema")
finished = True

with ThreadPoolExecutor(AI_WORKERS) as pool:
    for b0 in range(start_b, n_batches, AI_WORKERS):
        next_b = min(b0 + AI_WORKERS, n_batches)
        try:
            done = list(pool.map(classify_batch, range(b0, next_b)))
        except Exception as e:
            if not is_quota_error(e):
                raise
            save_checkpoint(b0)   # batch di putaran ini diulang saat lanjut
            print(f"[STOP] Kuota habis / rate limit di batch {b0}/{n_batches}. "
                  "Tunggu beberapa menit, lalu jalankan ulang cell ini (otomatis lanjut dari batch ini).")
            finished = False
            break

        for idx_batch, results in done:
            if results is None:
                print(f"[ERROR] 1 batch ({len(idx_batch)} cuitan) gagal total -> tetap tanpa tema.")
                continue
            for r, df_idx in zip(results, idx_batch):
                theme = r.get("primary_theme")
                df_clean.at[df_idx, "theme"] = theme if theme in THEMES else "lainnya"
                df_clean.at[df_idx, "theme_confidence"] = float(r.get("confidence", 0.5))

        pbar.update(next_b - b0)
        if next_b // CHECKPOINT_EVERY != b0 // CHECKPOINT_EVERY:
            save_checkpoint(next_b)
        time.sleep(SLEEP_BETWEEN_CALLS)
pbar.close()

if finished and os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

n_unprocessed = df_clean["theme"].isna().sum()
if n_unprocessed:
    print(f"[WARN] {n_unprocessed} baris BELUM punya tema (sampling / kuota habis / batch gagal). "
          "Laporan di bawah TIDAK lengkap.")

print("\nDistribusi tema:")
display(df_clean["theme"].value_counts(dropna=False))
print("\nContoh hasil klasifikasi:")
display(df_clean.loc[df_clean["theme"].isin(RELEVANT_THEMES), ["text_clean", "theme", "theme_confidence"]].head(10))
print(f"[OK] Total API call: {api_call_count}")

[INFO] Keyword yang paling banyak meloloskan baris: [('rusia', 126), ('geopolitik', 103), ('infrastruktur', 103), ('komoditas', 83), ('brics', 69), ('ukraina', 29), ('alutsista', 25), ('kesepakatan', 16), ('digitalisasi', 14), ('vladivostok', 12)]
[INFO] Prefilter: 20940 baris 'lainnya', 520 lanjut ke AI
[INFO] Sisa 13 batch, 1 paralel


Klasifikasi tema:   0%|          | 0/13 [00:00<?, ?it/s]


Distribusi tema:


,count
theme,
lainnya,21340
reaksi_geopolitik,85
kerja_sama_bilateral,30
agenda_pertemuan,5



Contoh hasil klasifikasi:


,text_clean,theme,theme_confidence
60,"Dinamika geopolitik Indonesia di panggung internasional kembali memanas. Langkah tegas Presiden Prabowo Subianto meneriakkan dukungan untuk kebebasan Palestina kini mendapat respons tajam dan sorotan kritis dari media Israel, The Times of Israel. Peristiwa ini berawal saat Presiden Prabowo menghadiri peringatan 100 tahun Pondok Modern Darussalam Gontor di Ponorogo, Jawa Timur, Sabtu (19/9/2026). Di hadapan ribuan peserta, Prabowo secara emosional menyapa Duta Besar Palestina untuk Indonesia, Abdulfattah Al-Sattari. Momen haru tercipta ketika Dubes Palestina mengalungkan syal keffiyeh kepada Prabowo. Sambil mengusap air matanya, Presiden menyampaikan permohonan maaf atas keharuannya. ""Saya minta maaf. Tadi saya teringat saudara-saudara kita di Palestina,"" kata Prabowo dalam sambutannya. Prabowo lantas mengenakan keffiyeh tersebut sepanjang acara sembari berulang kali meneriakkan seruan ""Free Palestine"". Sikap vokal tersebut langsung memicu reaksi media Israel, The Times of Israel. Mereka mengungkit kembali rekam jejak diplomasi Prabowo saat menghadiri Sidang Umum PBB pada September 2025. Dalam ulasannya, media tersebut menyindir perbedaan nada politik diplomasi Prabowo yang dinilai bergeser. ""Dalam pidato tersebut, Prabowo menyampaikan bahwa perdamaian membutuhkan pengakuan, penghormatan, serta jaminan terhadap keamanan Israel,"" tulis The Times of Israel dalam laporannya. Tak hanya itu, media tersebut turut memuat klaim dari sebuah sumber yang menyebut bahwa Prabowo sempat menyetujui rencana kunjungan ke Israel pasca-kesepakatan gencatan senjata di Gaza, sebelum akhirnya dibatalkan karena terlanjur bocor ke publik. Menanggapi gejolak isu ini, Kementerian Luar Negeri Republik Indonesia dengan tegas membantah seluruh spekulasi mengenai rencana kunjungan Presiden Prabowo ke Israel. Kemlu RI menegaskan kembali komitmen dan posisi konstan Indonesia yang tidak pernah goyah dalam mendukung kemerdekaan serta hak-hak rakyat Palestina. Jadikan //t.co/en8Xa9nh9s sumber berita Klik //t.co/Q6CsR9GSL6 & centang sebagai referensi pilihan. Aktifkan lonceng notifikasi & bagikan konten ini. prabowo palestina mediaisrael sindiran BM",reaksi_geopolitik,1.0
238,"Prabowo Tidak memahami prinsip NON BLOK dan prinsip Geopolitik BEBAS AKTIF secara Komprehensif, karena setiap Prilakunya selalu mengemban beban pribadi dirinya sendiri, pada KEPENTINGAN POLITIK PRAKTIS agar dirinya bisa bertahan untuk 2 periode, sejak awal dia dilantik. Maka dari itu prabowo telah membawa Bangsa dan Negara Indonesia kepada ARAH yang SALAH dalam banyak hal baik dari sisi Geopolitik, Geoekonomi juga dalam Kebijakannya politik dan ekonomi didalam negeri. Didalam negeri, prabowo lebih banyak Memerankan dirinya sebagai MAKELAR OLIGARKI daripada memerankan dirinya sebagai PRESIDEN.. Meski teriakannya setiap kesempatannya didepan MICROPHONE, Saya siap mati.. please JANGAN bapak melipat tangan , saya lebih berkenan, jika bapak BERTOBAT dahulu sebelum Mati diludahin Rakyat melipat tangan wajah berkacamata hitam",reaksi_geopolitik,0.9
254,"Dari awal prabowo itu tidak punya pendirian, mencla mencle, geng china rusia ikut, geng AS Israel ikut. Pemimpin yang tidak punya pendirian itu tidak punya harga diri, kewibawaan dan tidak akan bisa dipercaya. Tidak akan bisa membawa kejayaan, bisanya kehancuran. TERBUKTI, racun",reaksi_geopolitik,0.9
276,prabowo masuk media rusia - gara gara teriak free palestina beberapa hari yang lalu - setelah teriakan itu dubes palestina buat indonesia abdulfattah al sattari maju ke presiden terus pasangin syal khas palestina kufiyah ke bahu prabowo - tapi di pertanyakan konsistensi prabowo - soalnya masih gabung bop tapi teriak free palestina,reaksi_geopolitik,1.0
303,"Comparison Statement Analysis Presiden RI di EEF(Eastern Economic Forum) 3 Sept 2026, Vladivostok Russia. Yang dimuat di The Star dan The Straits Time. //t.co/b5ATZhtHjE",agenda_pertemuan,0.8
577,"Lu paham tidak sih substansi permasalahan nya, woi Ba

[OK] Total API call: 14


# Step 5 - Ranking Top Author per Tema

Hanya akun non-media (sudah difilter Step 2). Skor = post count + engagement + followers.

In [ ]:
processed = df_clean["theme"].notna()
relevant = processed & (df_clean["theme"] != "lainnya")
confident = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE

df_topic = df_clean[relevant & confident].copy()
print(f"[INFO] {len(df_topic)} baris ikut ranking, "
      f"{(relevant & ~confident).sum()} baris dibuang karena tema ragu (< {RANKING_MIN_CONFIDENCE})")
if (~processed).sum():
    print(f"[WARN] {(~processed).sum()} baris belum diproses. Ranking PARSIAL.")

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]
# dihitung ulang di sini (bukan dari Step 2) supaya tetap ada walau df_clean berasal dari checkpoint lama
df_topic["author_key"] = df_topic[COL_AUTHOR_ID].map(canonical)


def top_spelling(frame):
    """author_key -> ejaan Author yang paling sering muncul (label yang enak dibaca di laporan & Gephi)."""
    return (frame.groupby(["author_key", COL_AUTHOR_ID]).size().sort_values(ascending=False)
            .reset_index().drop_duplicates("author_key").set_index("author_key")[COL_AUTHOR_ID])


# retweet bukan karya author-nya (dan kolom Retweeted-nya milik tweet asli) -> tidak dihitung ke volume.
# pengaruhnya masuk ke penulis asli lewat jaringan di Step 5B.
df_original = df_topic[~df_topic["is_retweet"]]
print(f"[INFO] {len(df_topic) - len(df_original)} retweet tidak dihitung ke skor volume (masuk ke jaringan)")

# dikelompokkan per AKUN (author_key), bukan per ejaan: 'prabowo.subianto' (IG) + '@Prabowo_Subianto' = 1 akun
agg = (df_original.groupby(["theme", "author_key"])
       .agg(jumlah_post=("engagement", "size"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"))
       .reset_index()
       .rename(columns={"author_key": "Id"}))
agg["author_id"] = agg["Id"].map(top_spelling(df_original))
# tiap metrik dinormalisasi ke max-nya dalam tema yang sama
metrics = agg[["jumlah_post", "total_engagement", "followers"]].astype(float)
norm = metrics / metrics.groupby(agg["theme"]).transform("max").replace(0, 1)
agg["volume_score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]
print(f"[INFO] Skor volume dihitung untuk {len(agg)} pasangan akun-tema "
      f"({df_original[COL_AUTHOR_ID].nunique()} ejaan Author -> {df_original['author_key'].nunique()} akun)")

[INFO] 120 baris ikut ranking, 0 baris dibuang karena tema ragu (< 0.6)
[INFO] 7 retweet tidak dihitung ke skor volume (masuk ke jaringan)
[INFO] Skor volume dihitung untuk 95 pasangan akun-tema (93 ejaan Author -> 93 akun)


# Step 5B - Nodes & Edges + KOL gabungan (volume + jaringan)

In [ ]:
import numpy as np

HANDLE = re.compile(r"(?<![\w.])@([\w.]*\w)")   # dari extractor: handle bertitik (IG) ikut, e-mail tidak


def is_media_id(key):
    """Key akun media: dikenali lewat ejaan mana pun di Step 2 (MEDIA_IDS), atau dari key-nya sendiri."""
    return key in MEDIA_IDS or is_media_account(key, extra_accounts=EXTRA_MEDIA_ACCOUNTS)


def extract_targets(text, retweet_of=None):
    """Satu post -> maksimal satu edge per akun tujuan (key canonical). Prioritas: retweet > reply > mention.
    retweet_of = penulis asli untuk retweet yang tidak menulis @user (dicari lewat teks yang sama)."""
    rt = RE_RT.search(text)
    source = canonical(rt.group(1)) if rt else retweet_of
    targets = {source: "retweet"} if source else {}
    for m in RE_REPLY.findall(text):
        targets.setdefault(canonical(m), "reply")
    for m in HANDLE.findall(text):
        targets.setdefault(canonical(m), "mention")
    return list(targets.items())


def eigenvector_all(U):
    """Eigenvector di SETIAP klaster terhubung, bukan hanya giant component, jadi semua akun di jaringan dapat nilai.
    Tiap klaster dinormalisasi sendiri, lalu dikali (nilai eigen klaster / nilai eigen terbesar): klaster terkuat
    nilainya sama seperti versi giant-only, klaster kecil (mis. 2 akun saling mention sekali) jadi kecil, tidak
    menyamai pusat klaster besar."""
    parts = []
    for comp in nx.connected_components(U):
        nodes = list(comp)
        # ponytail: eigen-dekomposisi dense O(n^3); cukup cepat sampai beberapa ribu akun per klaster,
        # ganti ke scipy.sparse.linalg.eigsh kalau satu klaster jauh lebih besar dari itu.
        vals, vecs = np.linalg.eigh(nx.to_numpy_array(U, nodelist=nodes, weight="Weight"))
        parts.append((vals[-1], pd.Series(np.abs(vecs[:, -1]), index=nodes)))
    lam_max = max(lam for lam, _ in parts)
    return pd.concat([vec * lam / lam_max for lam, vec in parts])


def centrality(pair_w):
    """PageRank (graf berarah) + Eigenvector (graf tak berarah, semua klaster) per akun."""
    G = nx.from_pandas_edgelist(pair_w, "Source", "Target", edge_attr="Weight", create_using=nx.DiGraph)
    pair = np.sort(pair_w[["Source", "Target"]].to_numpy(), axis=1)   # A->B dan B->A dijumlah
    und = pair_w.assign(u=pair[:, 0], v=pair[:, 1]).groupby(["u", "v"], as_index=False)["Weight"].sum()
    U = nx.from_pandas_edgelist(und, "u", "v", edge_attr="Weight")
    out = pd.DataFrame({"pagerank": pd.Series(nx.pagerank(G, weight="Weight"))})
    out["eigenvector"] = eigenvector_all(U).reindex(out.index, fill_value=0.0)
    return out.rename_axis("Id").reset_index()


assert extract_targets("RT @Abc: halo @Def @abc. @def, email x@gmail.com") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("RT halo @def", retweet_of="abc") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("setuju banget [RE Abc]") == [("abc", "reply")]
assert extract_targets("cc @prabowo.subianto @Prabowo_Subianto") == [("prabowosubianto", "mention")]
# toy: 8 akun me-reply "hub" (klaster terbesar) + 2 akun yang saling mention sekali (klaster kecil)
_toy = centrality(pd.DataFrame({"Source": [f"r{i}" for i in range(8)] + ["x"],
                                "Target": ["hub"] * 8 + ["y"], "Weight": [1] * 9})).set_index("Id")["eigenvector"]
assert abs(_toy["hub"] - 0.7071) < 1e-3 and abs(_toy["r0"] - 0.25) < 1e-3     # sama dengan versi giant-only
assert 0 < _toy["x"] < _toy["hub"]                                             # klaster kecil: tidak 0, tapi di bawah pusat klaster besar
assert abs(centrality(pd.DataFrame({"Source": ["a", "b"], "Target": ["b", "c"], "Weight": [1, 1]}))["pagerank"].sum() - 1) < 1e-6

# --- merge akun: semua ejaan Author -> satu key (author_key dibuat di Step 2) ---
author_key = df_raw["author_key"]
merges = (df_raw.assign(Author=df_raw[COL_AUTHOR_ID].astype(str))
          .groupby("author_key")["Author"].agg(ejaan=lambda s: sorted(s.unique()), baris="size"))
merges = merges[merges["ejaan"].map(len) > 1].sort_values("baris", ascending=False)
print(f"[INFO] {df_raw[COL_AUTHOR_ID].nunique():,} ejaan Author -> {author_key.nunique():,} akun; "
      f"{len(merges):,} akun menggabungkan 2+ ejaan (cek di bawah; salah gabung -> perbaiki lewat ALIASES)")
display(merges)

# --- edges: dari df_raw (sebelum dedup) supaya setiap retweet terhitung ---
raw_keys = dedup_key(df_raw["text_clean"])
# tema diturunkan dari df_topic -> retweet ikut tema post aslinya, dengan syarat confidence yang sama
theme_by_key = dict(zip(dedup_key(df_topic["text_clean"]), df_topic["theme"]))
raw_theme = raw_keys.map(theme_by_key)
# "RT <teks>" tanpa @user: penulis asli = author post non-retweet dengan teks yang sama
orig_author = (pd.DataFrame({"key": raw_keys, "author": author_key})[~df_raw["is_retweet"]]
               .drop_duplicates("key").set_index("key")["author"])
retweet_of = raw_keys.map(orig_author).where(df_raw["is_retweet"])
print(f"[INFO] {df_raw['is_retweet'].sum()} retweet, {retweet_of.notna().sum()} di antaranya ketemu post aslinya di data")

edge_rows = pd.DataFrame([
    {"Source": author, "Target": target, "interaction": kind, "theme": theme, "text": text}
    for author, raw, text, theme, rt_of in zip(author_key, df_raw["text_raw_combined"],
                                               df_raw["text_clean"], raw_theme, retweet_of)
    if pd.notna(theme) and author is not None   # post tanpa Author tidak punya asal edge
    for target, kind in extract_targets(raw, rt_of if pd.notna(rt_of) else None)
    if target != author
], columns=["Source", "Target", "interaction", "theme", "text"])

# kolom 'Type' sengaja tidak dipakai: di Gephi itu kolom khusus Directed/Undirected
df_edges = edge_rows.groupby(["Source", "Target", "interaction", "theme"]).size().rename("Weight").reset_index()

# teks yang me-retweet X = kata-kata X sendiri -> dipakai Step 6B untuk KOL yang tidak posting di data
rt_posts = (edge_rows[edge_rows["interaction"] == "retweet"]
            .groupby(["Target", "theme"])["text"].apply(lambda s: s.drop_duplicates().tolist()).to_dict())

# --- PageRank + Eigenvector per tema ---
net_parts = [centrality(sub.groupby(["Source", "Target"], as_index=False)["Weight"].sum()).assign(theme=theme)
             for theme, sub in df_edges.groupby("theme")]
df_net = (pd.concat(net_parts, ignore_index=True) if net_parts
          else pd.DataFrame(columns=["Id", "pagerank", "eigenvector", "theme"]))

# --- gabung volume + jaringan ---
followers_by_id = df_raw.assign(Id=author_key).groupby("Id")[COL_FOLLOWERS].max()

kol = agg.merge(df_net, on=["theme", "Id"], how="outer")
# INCLUDE_MEDIA (Step 2) = False -> akun media tetap jadi node jaringan, tapi tidak boleh jadi KOL
if not INCLUDE_MEDIA:
    kol = kol[~kol["Id"].map(is_media_id)]
kol = kol.copy()
kol["is_media"] = kol["Id"].map(is_media_id)
kol["author_id"] = kol["author_id"].fillna(kol["Id"])                   # KOL yang cuma muncul di jaringan
kol["followers"] = kol["followers"].fillna(kol["Id"].map(followers_by_id))
score_cols = ["jumlah_post", "total_engagement", "followers", "volume_score", "pagerank", "eigenvector"]
kol[score_cols] = kol[score_cols].fillna(0)
# dinormalisasi ke max tema SETELAH filter media (kalau INCLUDE_MEDIA = False), supaya skor antar akun sebanding
norm_theme = lambda col: kol[col] / kol.groupby("theme")[col].transform("max").replace(0, 1)
kol["network_score"] = NET_WEIGHT_PAGERANK * norm_theme("pagerank") + NET_WEIGHT_EIGEN * norm_theme("eigenvector")
kol["score"] = KOL_WEIGHT_VOLUME * kol["volume_score"] + KOL_WEIGHT_NETWORK * kol["network_score"]

df_top_authors = (kol.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

# --- nodes (Id unik, format Gephi); PageRank & Eigenvector di sini dihitung atas seluruh jaringan ---
best = kol.sort_values("score").drop_duplicates("Id", keep="last").set_index("Id")
pair_all = df_edges.groupby(["Source", "Target"], as_index=False)["Weight"].sum()
top_ids = set(df_top_authors["Id"])
glob = centrality(pair_all).set_index("Id") if len(pair_all) else pd.DataFrame(columns=["pagerank", "eigenvector"])

df_nodes = pd.DataFrame({"Id": sorted(set(pair_all["Source"]) | set(pair_all["Target"]) | top_ids)})
df_nodes["Label"] = df_nodes["Id"].map(top_spelling(df_raw)).fillna(df_nodes["Id"])   # ejaan asli, bukan key
df_nodes["in_weight"] = df_nodes["Id"].map(pair_all.groupby("Target")["Weight"].sum()).fillna(0).astype(int)
df_nodes["out_weight"] = df_nodes["Id"].map(pair_all.groupby("Source")["Weight"].sum()).fillna(0).astype(int)
df_nodes["pagerank"] = df_nodes["Id"].map(glob["pagerank"]).fillna(0)
df_nodes["eigenvector"] = df_nodes["Id"].map(glob["eigenvector"]).fillna(0)
df_nodes["followers"] = df_nodes["Id"].map(followers_by_id)   # NaN = cuma muncul sebagai target
df_nodes["is_media"] = df_nodes["Id"].map(is_media_id)
df_nodes["kol_theme"] = df_nodes["Id"].map(best["theme"])
df_nodes["kol_score"] = df_nodes["Id"].map(best["score"]).fillna(0)
df_nodes["is_kol"] = df_nodes["Id"].isin(top_ids)

print(f"[OK] {len(df_nodes)} nodes, {len(df_edges)} edges, {len(df_top_authors)} KOL")
print(f"[INFO] Edge per interaksi:\n{df_edges.groupby('interaction')['Weight'].sum()}")

# cek kolom Author = username; kalau tidak, jaringan tidak nyambung ke skor volume
if len(df_edges):
    match_rate = df_edges["Target"].isin(followers_by_id.index).mean()
    print(f"[INFO] {match_rate:.0%} target edge juga muncul sebagai Author di data. "
          "Kalau mendekati 0%, cek apakah kolom Author berisi username (bukan nama tampilan).")

for theme in RELEVANT_THEMES:
    sub = df_top_authors[df_top_authors["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue
    print(f"\n=== KOL: {theme} ===")
    display(sub[["rank", "author_id", "is_media", "jumlah_post", "total_engagement", "followers",
                 "pagerank", "eigenvector", "volume_score", "network_score", "score"]])

print("\n=== Nodes (urut PageRank) ===")
display(df_nodes.sort_values("pagerank", ascending=False).reset_index(drop=True))
print("\n=== Edges (urut Weight) ===")
display(df_edges.sort_values("Weight", ascending=False).reset_index(drop=True))

[INFO] 17,217 ejaan Author -> 17,104 akun; 65 akun menggabungkan 2+ ejaan (cek di bawah; salah gabung -> perbaiki lewat ALIASES)


,ejaan,baris
author_key,,
,"[....._....., ÃÂ§ÃÂ¤ÃÂÃÂÃÂ§ÃÂ¡, ÃÂ¨ÃÂ§ÃÂÃÂÃÂ³ÃÂ·, ÃÂ²ÃÂÃÂ§ ÃÂ´ÃÂ§ÃÂÃÂÃÂªÃÂ±ÃÂ§, ÃÂºÃÂÃÂÃÂÃÂ, ÃÂÃÂÃÂ¨ÃÂ ÃÂÃÂ§ÃÂ ÃÂÃÂÃÂ§, Ã Â¸ÂÃ Â¸Â§Ã Â¸Â²Ã Â¸Â¡Ã Â¸Â¥Ã Â¸Â±Ã Â¸ÂÃ Â¸ÂÃ Â¸ÂµÃ Â¹ÂÃ Â¹ÂÃ Â¸Â¡Ã Â¹ÂÃ Â¹ÂÃ Â¸ÂÃ Â¸Â´Ã Â¸ÂÃ Â¹ÂÃ Â¸ÂÃ Â¸Â¢Ã Â¸ÂÃ , Ã¡Â´ÂÃÂÃªÂÂ°ÃÂªÃ¡Â´ÂÃÂ´ Ã¡Â´Â¡ÃÂªÃ¡Â´Â ÃÂªÃ¡Â´ÂÃ¡Â´ÂÃ¡Â´ÂÃ¡Â´ÂÃ°ÂÂÂµÃ°ÂÂÂ¸, Ã¡Â´ÂÃÂªÃÂÃÂÃ¡Â´ÂÃÂ´ Ã¡Â´Â ÃÂªÃ¡Â´Â ÃÂªÃ¡Â´ÂÃÂ´Ã¡Â´Â, Ã¢ÂÂ¨, Ã£ÂÂÃ£ÂÂÃ£ÂÂ¢, Ã¥Â¾Â·Ã¨ÂÂ£, Ã§ÂÂ§Ã¦ÂÂÃ°ÂÂÂ®Ã°ÂÂÂ© Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂ-Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¦Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¤ Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¬Ã°ÂÂÂ­ | Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ, Ã°ÂÂÂ©Ã°ÂÂÂ¶Ã°ÂÂÂ¹Ã°ÂÂÂ¾Ã°ÂÂÂ¶ Ã°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¿Ã°ÂÂÂ¶, Ã°ÂÂÂ«Ã°ÂÂÂÃ°ÂÂÂ¿Ã°ÂÂÂ¾, Ã°ÂÂÂÃ°ÂÂÂ®Ã°ÂÂÂ°Ã°ÂÂÂª Ã°ÂÂÂÃ°ÂÂÂ¾Ã°ÂÂÂ½Ã°ÂÂÂ»Ã°ÂÂÂ² Ã°ÂÂÂÃ°ÂÂÂ·Ã°ÂÂÂ°Ã°ÂÂÂ®Ã°ÂÂÂµÃ°ÂÂÂ²Ã°ÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¥Ã°ÂÂÂÃ°ÂÂÂ§Ã°ÂÂÂÃ°ÂÂÂ Ã¢ÂÂÃ°ÂÂÂ¦Ã°ÂÂÂ§Ã°ÂÂÂÃ°ÂÂÂ¥Ã°ÂÂÂÃ°ÂÂÂ¤Ã°ÂÂÂÃ°ÂÂÂ£Ã, Ã°ÂÂÂ»Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ & Ã°ÂÂÂ®Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ§Ã°ÂÂÂ©Ã°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂÃ°ÂÂÂ¤Ã°ÂÂÂ£Ã°ÂÂÂÃ°ÂÂÂ¡, Ã°ÂÂÂ³Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ£Ã°ÂÂÂ, Ã°ÂÂÂ½Ã°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ, Ã°ÂÂÂ, Ã°ÂÂÂ»Ã°ÂÂÂ»Ã°ÂÂÂ», Ã°ÂÂ¤Â , محب الخير , ც𝓵𝒶𝒸𝓀',𝓞𝔀𝓵^, ◟♯ . / kᥱіk᥆ •ᗢ, 大老远啧啧, 멍이, 𝐀𝐬𝐚𝐥𝐁𝐞𝐫𝐬𝐮𝐚, 𝐊𝐢𝐜𝐚𝐮 𝐆𝐚𝐫𝐮𝐝𝐚, 𝐓𝐎𝐊𝐎 𝐊𝐃𝐌, 𝑦𝑎𝑛𝑧 𝑫 𝒏𝒚𝒂𝒘𝒊𝒕, 𝑩𝑨𝑵𝑮 𝑨𝑮𝑼𝑺 𝑪𝑯𝑨𝑵𝑵𝑬𝑳 , 𝓒𝓐𝓟𝓨 & 𝓟𝓪𝓷𝓭𝔂, 𝓗𝓪𝓷𝓷𝓲🦦ྀི , 𝕾𝖊𝖔𝖔_𝖌𝖗𝖆𝖞𝖞, 𝗠𝗨𝗞𝗘𝗫 𝗦𝗧𝗢𝗥𝗬, 𝗦𝗮𝗿𝗲_𝗩𝗹𝗼𝗴𝘀𝘀𝘀𐙚, 𝙏𝙀𝙈𝘼𝙉 𝙈𝙊𝙇𝙀 𝙄𝙉𝘿, 𝙒𝙝𝙞𝙯 𝙄𝘿 𝘾𝙝𝙖𝙣𝙣𝙚𝙡, 𝙺𝙰𝙽𝙶 𝙵𝙰𝙳𝙾𝙻𝙸 𝙱𝙰𝙽𝚃𝙴𝙽, 𝚍𝚒𝚜𝚔𝚘𝚗𝚋𝚘𝚡, 𝚣𝚘𝚗𝚊 𝚜𝚑𝚘𝚛𝚝𝚜]",59
sorottanahair,"[SOROT TANAH AIR, Sorot Tanah Air]",23
dialectique,"[DIALECTIQUE, Dialectique]",21
totalpolitik,"[Total Politik, totalpolitik]",18
dekade08,"[@dekade_08, DEKADE 08, Dekade 08, dekade_08]",14
katadataindonesia,"[KATADATA Indonesia, Katadata Indonesia]",12
maswanto,"[MAS WANTO, Mas Wanto]",12
tvrinasional,"[@TVRINasional, TVRI Nasional, tvrinasional]",10
ahmadkhozinudin,"[AHMAD KHOZINUDIN, Ahmad Khozinudin]",9


[INFO] 13816 retweet, 4043 di antaranya ketemu post aslinya di data
[OK] 114 nodes, 97 edges, 24 KOL
[INFO] Edge per interaksi:
interaction
mention    51
reply      17
retweet    30
Name: Weight, dtype: int64
[INFO] 64% target edge juga muncul sebagai Author di data. Kalau mendekati 0%, cek apakah kolom Author berisi username (bukan nama tampilan).

=== KOL: agenda_pertemuan ===


,rank,author_id,is_media,jumlah_post,total_engagement,followers,pagerank,eigenvector,volume_score,network_score,score
0,1,troublemakerid,False,1.0,0.0,1562.0,0.0,0.0,0.750000,0.0,0.375000
1,2,@bambangpe,False,1.0,0.0,167.0,0.0,0.0,0.616037,0.0,0.308019
2,3,Akhand Bharat,False,1.0,0.0,0.0,0.0,0.0,0.600000,0.0,0.300000
3,4,Meta AI,False,1.0,0.0,0.0,0.0,0.0,0.600000,0.0,0.300000



=== KOL: kerja_sama_bilateral ===


...

,rank,author_id,is_media,jumlah_post,total_engagement,followers,pagerank,eigenvector,volume_score,network_score,score
14,1,@LambeSahamjja,False,2.0,82.0,85254.0,0.208501,0.078812,0.700000,0.733964,0.716982
15,2,Meta AI,False,4.0,0.0,0.0,0.011930,0.000496,0.600000,0.040265,0.320133
16,3,Dedi,False,4.0,0.0,16.0,0.008372,0.002952,0.600028,0.029379,0.314703
17,4,@oconXBoyy,False,1.0,7.0,3583.0,0.008372,0.696144,0.177646,0.328107,0.252876
18,5,@Sebelahkampus,False,1.0,8.0,64691.0,0.008372,0.265507,0.288211,0.142526,0.215368
19,6,@GlendMarti35542,False,2.0,0.0,1103.0,0.008372,0.098781,0.301941,0.070676,0.186308
20,7,@TaliUdeng,False,1.0,50.0,24604.0,0.000000,0.000000,0.345728,0.000000,0.172864
21,8,@GarudaGenz19371,False,2.0,2.0,2.0,0.000000,0.000000,0.306101,0.000000,0.153051
22,9,kliper pemerintah,False,2.0,0.0,0.0,0.000000,0.000000,0.300000,0.000000,0.150000
23,10,nextalk,False,2.0,0.0,0.0,0.000000,0.000000,0.300000,0.000000,0.150000



=== Nodes (urut PageRank) ===


,Id,Label,in_weight,out_weight,pagerank,eigenvector,followers,is_media,kol_theme,kol_score,is_kol
0,lambesahamjja,@LambeSahamjja,29,0,0.164373,0.078812,85326.0,False,reaksi_geopolitik,0.716982,True
1,thisismeonx,thisismeonx,8,0,0.051478,0.335843,NaN,False,kerja_sama_bilateral,0.500000,True
2,mhuseinali,@mhuseinali,1,0,0.026516,0.118738,24541.0,False,kerja_sama_bilateral,0.233302,False
3,admkuznetsov45,@admkuznetsov45,3,1,0.023429,0.237477,1158.0,False,kerja_sama_bilateral,0.565491,True
4,youtube,youtube,1,0,0.012210,0.118738,NaN,False,reaksi_geopolitik,0.051583,False
5,theeconomicinfluenceid,theeconomicinfluenceid,1,0,0.012210,0.118738,NaN,False,kerja_sama_bilateral,0.136045,False
6,pororo266,@Pororo266,1,0,0.012210,0.118738,166.0,False,kerja_sama_bilateral,0.498564,True
7,rudetiawanst,rudetiawanst,1,0,0.012210,0.118738,NaN,False,reaksi_geopolitik,0.051583,False
8,theinsiderpaper,theinsiderpaper,1,0,0.012210,0.118738,NaN,False,reaksi_geopolitik,0.051583,False
9,hippoacademyid,hippoacademyid,1,0,0.012210,0.118738,NaN,False,reaksi_geopolitik,0.051583,False



=== Edges (urut Weight) ===


,Source,Target,interaction,theme,Weight
0,glendmarti35542,prabowo,mention,reaksi_geopolitik,2
1,4rsyat4ri,thisismeonx,reply,kerja_sama_bilateral,1
2,admkuznetsov45,mhuseinali,reply,kerja_sama_bilateral,1
3,aliam36,youtube,mention,reaksi_geopolitik,1
4,aiselif,theinsiderpaper,reply,reaksi_geopolitik,1
5,amatt,lambesahamjja,retweet,reaksi_geopolitik,1
6,annewillsjoy,lambesahamjja,retweet,reaksi_geopolitik,1
7,babingepet01nk,admkuznetsov45,retweet,kerja_sama_bilateral,1
8,alqossam3,admkuznetsov45,retweet,kerja_sama_bilateral,1
9,citraayuning7,thisismeonx,reply,kerja_sama_bilateral,1


# Step 6A: Load LLM lokal

In [ ]:
import ctypes, glob, site

# wheel llama-cpp (cu124) butuh library CUDA 12, sedangkan Colab sekarang CUDA 13 -> cari/pasang lalu muat duluan.
CUDA12_LIBS = ["libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"]   # urutan penting: cublasLt sebelum cublas

def find_lib(name):
    roots = {*site.getsitepackages(), *[p for p in sys.path if os.path.isdir(p)]}
    hits = [p for r in roots for p in glob.glob(f"{r}/nvidia/**/{name}", recursive=True)]
    return hits[0] if hits else None

if torch.cuda.is_available() and torch.version.cuda.startswith("13"):
    if not all(map(find_lib, CUDA12_LIBS)):
        !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    missing_libs = [n for n in CUDA12_LIBS if not find_lib(n)]
    if missing_libs:
        raise RuntimeError(f"Library CUDA 12 tetap tidak ditemukan: {missing_libs}. Kirim output cell ini untuk dicek.")
    for name in CUDA12_LIBS:
        ctypes.CDLL(find_lib(name), mode=ctypes.RTLD_GLOBAL)

# import yang gagal sebelumnya bisa tertinggal setengah jadi -> buang supaya import ulang dari awal
for m in [m for m in sys.modules if m.startswith("llama_cpp")]:
    del sys.modules[m]

import llama_cpp
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

if not torch.cuda.is_available():
    print("⚠️ Tidak ada GPU: LLM 9B di CPU bisa beberapa menit per KOL. "
          "Runtime -> Change runtime type -> T4 GPU, lalu mulai lagi dari cell Install.")
elif not llama_cpp.llama_supports_gpu_offload():
    print("⚠️ llama-cpp terpasang TANPA dukungan GPU -> LLM akan jalan di CPU (sangat lambat). "
          "Kirim output cell Install & cell ini untuk dicek.")

# cell ini dijalankan ulang -> pakai model yang sudah ada; load kedua kali = 2x memori GPU (bisa out of memory)
if "llm" in globals():
    print(f"[INFO] LLM lokal sudah ter-load, dipakai ulang: {LLM_MODEL_FILE}")
else:
    # download ~5,8 GB sekali per session (tersimpan di cache Colab selama session masih hidup)
    model_path = hf_hub_download(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE)
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=LLM_N_CTX, n_batch=512, verbose=False)
    print(f"[OK] LLM lokal siap: {LLM_MODEL_FILE} | GPU offload: {llama_cpp.llama_supports_gpu_offload()}")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): reconstructing file:   0%|          |  0.00B / 5.76GB            

gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[OK] LLM lokal siap: gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf | GPU offload: True


# Step 6B: Gemma menentukan sentimen, stance, dan summary (satu panggilan per KOL)


In [ ]:
LLM_TEST_N = None   # angka = coba beberapa KOL dulu; None = semua KOL

SENTIMENTS = ("positif", "negatif", "netral", "kontroversi")
STANCES = ("oposisi", "moderat", "supporter")
UNCLASSIFIED = "tidak_terklasifikasi"
KOL_FIELDS = ["relevan", "summary", "sentiment", "stance", "alasan"]
KOL_SCHEMA = {
    "type": "object",
    "properties": {"relevan": {"type": "boolean"},
                   "summary": {"type": "string"},
                   "sentiment": {"type": "string", "enum": list(SENTIMENTS)},
                   "stance": {"type": "string", "enum": list(STANCES)},
                   "alasan": {"type": "string"}},
    "required": KOL_FIELDS,
}

KOL_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}

Cuitan dari akun "{author}" tentang tema ini:
{posts}

Tugas:
1. relevan: true kalau cuitan benar-benar membahas topik di atas; false kalau membahas peristiwa lain
   dan hanya kebetulan menyebut nama tokoh, negara, atau acara yang sama.
2. summary: 2-3 kalimat tentang pandangan atau narasi utama akun ini soal tema tersebut.
   Jelaskan isinya, jangan menyalin cuitan kata per kata.
3. sentiment: SIKAP akun terhadap topik/tema, BUKAN nada bahasanya:
   - positif: mendukung, memuji, atau membela
   - negatif: mengkritik, menolak, atau menyindir
   - netral: hanya menyampaikan informasi, tanpa sikap yang jelas
   - kontroversi: dukungan dan kritik sama kuat
   Contoh: membela Prabowo sambil memaki pengkritiknya = positif (kasar ke pihak lain tidak mengubah sikapnya).
4. stance: sikap akun terhadap {target}, pilih satu:
   - supporter: mendukung, memuji, atau membela {target}
   - oposisi: mengkritik, menolak, atau menyerang {target}
   - moderat: hanya menyampaikan informasi, seimbang, atau kritik yang tidak menolak {target}
   Sentimen positif terhadap tema belum tentu berarti mendukung {target}, dan sebaliknya.
5. alasan: 1-2 kalimat alasan sentiment dan stance, berdasarkan isi cuitan.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def kol_posts(key, theme):
    """Post asli akun ini (semua ejaan) di tema ini, paling berpengaruh duluan. Tidak ada -> teks yang me-retweet dia."""
    own = (df_original[(df_original["author_key"] == key) & (df_original["theme"] == theme)]
           .sort_values("engagement", ascending=False)["text_clean"].dropna().tolist())
    return (own or rt_posts.get((key, theme), []))[:MAX_POSTS_PER_AUTHOR_SUMMARY]


def fmt_posts(posts, max_words=120):
    return "\n".join(f"{i}. " + " ".join(str(p).split()[:max_words]) for i, p in enumerate(posts, 1))


def pick(value, allowed):
    """Label dari LLM -> label baku; di luar daftar tidak ditebak."""
    value = str(value or "").strip().lower()
    return value if value in allowed else UNCLASSIFIED


def judge_kol(row, posts):
    """Satu panggilan LLM lokal -> relevan, summary, sentiment, stance, alasan."""
    content = KOL_PROMPT.format(topic=TOPIC, theme=row["theme"].replace("_", " "), target=STANCE_TARGET,
                                author=row["author_id"], posts=fmt_posts(posts))
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],   # Gemma tidak punya role 'system'
        response_format={"type": "json_object", "schema": KOL_SCHEMA},
        temperature=0.2, max_tokens=700,
    )
    out = json.loads(r["choices"][0]["message"]["content"])
    return {"relevan": bool(out.get("relevan", True)), "summary": str(out.get("summary") or "").strip(),
            "sentiment": pick(out.get("sentiment"), SENTIMENTS), "stance": pick(out.get("stance"), STANCES),
            "alasan": str(out.get("alasan") or "").strip()}


kol_texts = {}   # (Id, theme) -> post yang dibaca LLM; dipakai lagi oleh validasi Gemini di Step 6C
records = []
todo = df_top_authors if LLM_TEST_N is None else df_top_authors.head(LLM_TEST_N)
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Sentimen + stance + summary (LLM lokal)"):
    posts = kol_posts(row["Id"], row["theme"])
    if not posts:   # KOL yang cuma muncul di jaringan dan tidak punya teks sama sekali
        continue
    kol_texts[(row["Id"], row["theme"])] = posts
    try:
        out = judge_kol(row, posts)
    except Exception as e:
        print(f"[ERROR] {row['author_id']} | {row['theme']}: {type(e).__name__}: {e}")
        out = {"relevan": None, "summary": "", "sentiment": UNCLASSIFIED, "stance": UNCLASSIFIED,
               "alasan": f"error LLM: {type(e).__name__}"}
    records.append({**row[["theme", "rank", "author_id", "Id", "is_media", "score", "volume_score",
                           "network_score", "pagerank", "eigenvector"]].to_dict(), "jumlah_teks": len(posts), **out})

df_kol = pd.DataFrame(records)
print(f"[OK] {len(df_kol)} KOL dinilai LLM lokal ({len(todo) - len(df_kol)} KOL dilewati karena tidak punya teks)")
off_topic = df_kol["relevan"].eq(False)
if off_topic.any():
    print(f"[WARN] {off_topic.sum()} KOL dinilai TIDAK membahas topik (salah tema di Step 4): "
          f"{', '.join(df_kol.loc[off_topic, 'author_id'].astype(str))}. Tidak dihitung di rekap & ringkasan tema.")
if LLM_TEST_N is not None:
    print(f"[TEST] Baru {len(df_kol)} KOL. Kalau hasilnya sudah pas, set LLM_TEST_N = None lalu jalankan ulang cell ini.")
display(df_kol)

Sentimen + stance + summary (LLM lokal):   0%|          | 0/24 [00:00<?, ?it/s]

[OK] 23 KOL dinilai LLM lokal (1 KOL dilewati karena tidak punya teks)
[WARN] 1 KOL dinilai TIDAK membahas topik (salah tema di Step 4): @Sebelahkampus. Tidak dihitung di rekap & ringkasan tema.


,theme,rank,author_id,Id,is_media,score,volume_score,network_score,pagerank,eigenvector,jumlah_teks,relevan,summary,sentiment,stance,alasan
0,agenda_pertemuan,1,troublemakerid,troublemakerid,False,0.375000,0.750000,0.000000,0.000000,0.000000,1,True,"Akun ini memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, menekankan pidato Prabowo tentang politik luar negeri bebas aktif dan komitmen program domestik Indonesia.",positif,supporter,"Akun ini memuji kunjungan Prabowo dan pidatonya, serta menekankan komitmen program domestik Indonesia."
1,agenda_pertemuan,2,@bambangpe,bambangpe,False,0.308019,0.616037,0.000000,0.000000,0.000000,1,True,"Akun ini membagikan artikel dari The Star dan The Straits Times yang membahas pernyataan Presiden RI di EEF 2026 di Vladivostok, Rusia.",netral,moderat,"Akun ini hanya membagikan artikel tanpa komentar atau sikap yang jelas, hanya menyampaikan informasi tentang pernyataan Presiden RI di EEF 2026."
2,agenda_pertemuan,3,Akhand Bharat,akhandbharat,False,0.300000,0.600000,0.000000,0.000000,0.000000,1,True,"Akun ini membagikan foto empat pemimpin BRICS (Brasil, Rusia, India, Tiongkok, dan Afrika Selatan) sedang berdiskusi, termasuk Presiden Prabowo. Akun ini menyebut momen ini sebagai 'rare candid moment' atau momen langka dan tidak resmi.",netral,moderat,Akun ini hanya membagikan foto dan tidak menyampaikan pendapat atau kritik terhadap Presiden Prabowo atau pemerintah.
3,agenda_pertemuan,4,Meta AI,metaai,False,0.300000,0.600000,0.000000,0.000000,0.000000,1,True,"Akun ini membandingkan gaya kerja Presiden Prabowo dengan Presiden Jokowi dalam hal pendampingan saat kunjungan luar negeri. Prabowo selalu didampingi Seskab Teddy Indra Wijaya, sedangkan Jokowi lebih selektif dalam memilih pendamping.",netral,moderat,Akun ini hanya membandingkan gaya kerja Prabowo dan Jokowi tanpa menunjukkan sikap positif atau negatif terhadap keduanya.
4,kerja_sama_bilateral,1,@admkuznetsov45,admkuznetsov45,False,0.565491,0.600261,0.530721,0.110705,0.500000,1,True,"Akun ini membahas upaya pemerintah Indonesia dalam meningkatkan ketahanan energi, seperti program konversi minyak sawit menjadi diesel B50, impor minyak mentah dari Rusia, dan RDMP Kilang Dumai.",netral,moderat,Akun ini hanya menyampaikan informasi tentang upaya pemerintah dalam meningkatkan ketahanan energi tanpa menunjukkan sikap positif atau negatif terhadap Presiden Prabowo dan pemerintah.
5,kerja_sama_bilateral,3,@Pororo266,pororo266,False,0.498564,0.725037,0.272091,0.057691,0.250000,1,True,"Akun ini memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, serta mendorong kerja sama bilateral antara Indonesia dan Rusia dalam berbagai bidang seperti pelayaran, penerbangan, perdagangan, dan ekonomi.",positif,supporter,Akun ini memuji kunjungan Presiden Prabowo dan mendorong kerja sama bilateral antara Indonesia dan Rusia.
6,kerja_sama_bilateral,4,@latipiaa,latipiaa,False,0.425016,0.850032,0.000000,0.000000,0.000000,1,True,Akun ini mendukung upaya Presiden Prabowo untuk memperkuat kerja sama bilateral dengan negara-negara BRICS dalam hal industrialisasi dan rantai pasok global.,positif,supporter,Akun ini memuji upaya Presiden Prabowo untuk memperkuat kerja sama bilateral dengan negara-negara BRICS dalam hal industrialisasi dan rantai pasok global.
7,kerja_sama_bilateral,5,podcast_somestory,podcastsomestory,False,0.398063,0.600318,0.195809,0.031184,0.250000,1,True,Akun ini membahas rencana kerja sama antara Indonesia dan Rusia dalam bidang hilirisasi bauksit dan produksi pupuk.,netral,moderat,Akun ini hanya menyampaikan informasi tentang rencana kerja sama Indonesia-Rusia tanpa menunjukkan sikap positif atau negatif terhadap Presiden Prabowo dan pemerintah.
8,kerja_sama_bilateral,6,@sabila_tatihar,sabilatatihar,False,0.397904,0.600000,0.195809,0.031184,0.250000,1,True,"Akun ini mendukung kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin, karena dianggap penting untuk mengamankan energi bagi Ind

# Step 6C: Validasi Gemini (cell BARU)


In [ ]:
def build_validation_prompt(rows):
    # cuitan pakai "-" menjorok, supaya penomoran hanya untuk akun (field 'index' tidak tertukar)
    posts = lambda r: "\n".join("     - " + " ".join(str(p).split()[:120]) for p in kol_texts[(r.Id, r.theme)])
    items = "\n\n".join(
        f'{i}. Akun "{r.author_id}" | tema: {r.theme.replace("_", " ")}\n'
        f"   Cuitan:\n{posts(r)}\n"
        f"   Hasil model lokal -> sentiment: {r.sentiment}, stance: {r.stance}. Alasan: {r.alasan}"
        for i, r in enumerate(rows.itertuples(), 1)
    )
    return f"""Kamu validator analisis opini publik media sosial Indonesia. Topik: {TOPIC}.
Pihak yang dinilai untuk stance: {STANCE_TARGET}.

Definisi:
- sentiment = SIKAP akun terhadap topik/tema (bukan nada bahasa): positif, negatif, netral, kontroversi.
  Membela tokoh sambil memaki pengkritiknya tetap positif.
- stance = sikap terhadap {STANCE_TARGET}: supporter (mendukung/membela), oposisi (mengkritik/menolak),
  moderat (informatif, seimbang, atau kritik yang tidak menolak).

Periksa hasil model lokal untuk tiap akun berdasarkan cuitannya. Kalau sudah benar, ulangi labelnya;
kalau salah, ganti dengan label yang benar.

{items}

Jawab HANYA JSON array berisi tepat {len(rows)} objek dengan field:
index (nomor akun), sentiment, stance, catatan (alasan singkat kalau mengoreksi; string kosong kalau setuju)."""


def apply_validation(rows, results):
    """Gabungkan hasil Gemini ke df_kol. Label Gemini yang tidak valid -> tetap pakai label model lokal."""
    for (df_idx, row), r in zip(rows.iterrows(), results):
        sent = r.get("sentiment") if r.get("sentiment") in SENTIMENTS else row["sentiment"]
        stance = r.get("stance") if r.get("stance") in STANCES else row["stance"]
        changed = (sent, stance) != (row["sentiment"], row["stance"])
        df_kol.loc[df_idx, ["sentiment_final", "stance_final", "validasi", "catatan_gemini"]] = [
            sent, stance, "dikoreksi" if changed else "setuju", r.get("catatan", "") if changed else ""]


df_kol["sentiment_final"] = df_kol["sentiment"]   # default: hasil model lokal, ditimpa kalau Gemini mengoreksi
df_kol["stance_final"] = df_kol["stance"]
df_kol["validasi"] = "belum_divalidasi"
df_kol["catatan_gemini"] = ""

to_check = df_kol[df_kol["relevan"].ne(False) & df_kol["sentiment"].ne(UNCLASSIFIED)]
quota_hit = False
for b in tqdm(range(0, len(to_check), BATCH_SIZE_VALIDATE), desc="Validasi Gemini"):
    rows = to_check.iloc[b:b + BATCH_SIZE_VALIDATE]
    try:
        results = ask_json(build_validation_prompt(rows), parse=lambda p: order_by_index(p, len(rows)), retry=3)
    except Exception as e:
        if not is_quota_error(e):
            raise
        quota_hit = True
        break
    if results is not None:
        apply_validation(rows, results)
    time.sleep(SLEEP_BETWEEN_CALLS)

counts = df_kol["validasi"].value_counts()
print(f"[OK] Validasi Gemini: {counts.get('setuju', 0)} setuju, {counts.get('dikoreksi', 0)} dikoreksi, "
      f"{counts.get('belum_divalidasi', 0)} belum divalidasi. Total API call sesi ini: {api_call_count}")
if quota_hit:
    print("[STOP] Kuota Colab AI habis. KOL yang belum divalidasi tetap memakai hasil model lokal. "
          "Tunggu beberapa menit lalu jalankan ulang cell ini kalau mau semuanya divalidasi.")

print("\n=== Sentimen, stance & summary per KOL (final) ===")
display(df_kol[["theme", "rank", "author_id", "is_media", "relevan", "score", "volume_score", "network_score",
                "pagerank", "eigenvector", "sentiment_final", "stance_final", "validasi", "summary", "alasan",
                "catatan_gemini", "sentiment", "stance"]])

print(f"\n=== Rekap stance terhadap {STANCE_TARGET} per tema ===")
on_topic = df_kol[df_kol["relevan"].ne(False)]
stance_counts = (pd.crosstab(on_topic["theme"], on_topic["stance_final"])
                 .reindex(columns=[*STANCES, UNCLASSIFIED], fill_value=0))
display(stance_counts.loc[:, stance_counts.any()].assign(total=stance_counts.sum(axis=1)))

Validasi Gemini:   0%|          | 0/5 [00:00<?, ?it/s]

[OK] Validasi Gemini: 20 setuju, 2 dikoreksi, 1 belum divalidasi. Total API call sesi ini: 19

=== Sentimen, stance & summary per KOL (final) ===


,theme,rank,author_id,is_media,relevan,score,volume_score,network_score,pagerank,eigenvector,sentiment_final,stance_final,validasi,summary,alasan,catatan_gemini,sentiment,stance
0,agenda_pertemuan,1,troublemakerid,False,True,0.375000,0.750000,0.000000,0.000000,0.000000,positif,supporter,setuju,"Akun ini memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, menekankan pidato Prabowo tentang politik luar negeri bebas aktif dan komitmen program domestik Indonesia.","Akun ini memuji kunjungan Prabowo dan pidatonya, serta menekankan komitmen program domestik Indonesia.",,positif,supporter
1,agenda_pertemuan,2,@bambangpe,False,True,0.308019,0.616037,0.000000,0.000000,0.000000,netral,moderat,setuju,"Akun ini membagikan artikel dari The Star dan The Straits Times yang membahas pernyataan Presiden RI di EEF 2026 di Vladivostok, Rusia.","Akun ini hanya membagikan artikel tanpa komentar atau sikap yang jelas, hanya menyampaikan informasi tentang pernyataan Presiden RI di EEF 2026.",,netral,moderat
2,agenda_pertemuan,3,Akhand Bharat,False,True,0.300000,0.600000,0.000000,0.000000,0.000000,netral,moderat,setuju,"Akun ini membagikan foto empat pemimpin BRICS (Brasil, Rusia, India, Tiongkok, dan Afrika Selatan) sedang berdiskusi, termasuk Presiden Prabowo. Akun ini menyebut momen ini sebagai 'rare candid moment' atau momen langka dan tidak resmi.",Akun ini hanya membagikan foto dan tidak menyampaikan pendapat atau kritik terhadap Presiden Prabowo atau pemerintah.,,netral,moderat
3,agenda_pertemuan,4,Meta AI,False,True,0.300000,0.600000,0.000000,0.000000,0.000000,netral,moderat,setuju,"Akun ini membandingkan gaya kerja Presiden Prabowo dengan Presiden Jokowi dalam hal pendampingan saat kunjungan luar negeri. Prabowo selalu didampingi Seskab Teddy Indra Wijaya, sedangkan Jokowi lebih selektif dalam memilih pendamping.",Akun ini hanya membandingkan gaya kerja Prabowo dan Jokowi tanpa menunjukkan sikap positif atau negatif terhadap keduanya.,,netral,moderat
4,kerja_sama_bilateral,1,@admkuznetsov45,False,True,0.565491,0.600261,0.530721,0.110705,0.500000,positif,supporter,dikoreksi,"Akun ini membahas upaya pemerintah Indonesia dalam meningkatkan ketahanan energi, seperti program konversi minyak sawit menjadi diesel B50, impor minyak mentah dari Rusia, dan RDMP Kilang Dumai.",Akun ini hanya menyampaikan informasi tentang upaya pemerintah dalam meningkatkan ketahanan energi tanpa menunjukkan sikap positif atau negatif terhadap Presiden Prabowo dan pemerintah.,"Cuitan ini secara aktif menjelaskan dan membela berbagai langkah positif pemerintah dalam ketahanan energi, serta menyiratkan bahwa masyarakat mungkin kurang informasi, yang menunjukkan sikap mendukung pemerintah.",netral,moderat
5,kerja_sama_bilateral,3,@Pororo266,False,True,0.498564,0.725037,0.272091,0.057691,0.250000,positif,supporter,setuju,"Akun ini memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, serta mendorong kerja sama bilateral antara Indonesia dan Rusia dalam berbagai bidang seperti pelayaran, penerbangan, perdagangan, dan ekonomi.",Akun ini memuji kunjungan Presiden Prabowo dan mendorong kerja sama bilateral antara Indonesia dan Rusia.,,positif,supporter
6,kerja_sama_bilateral,4,@latipiaa,False,True,0.425016,0.850032,0.000000,0.000000,0.000000,positif,supporter,setuju,Akun ini mendukung upaya Presiden Prabowo untuk memperkuat kerja sama bilateral dengan negara-negara BRICS dalam hal industrialisasi dan rantai pasok global.,Akun ini memuji upaya Presiden Prabowo untuk memperkuat kerja sama bilateral dengan negara-negara BRICS dalam hal industrialisasi dan rantai pasok global.,,positif,supporter
7,kerja_sama_bilateral,5,podcast_somestory,False,True,0.398063,0.600318,0.195809,0.031184,0.250000,netral,moderat,setuju,Akun ini membahas rencana kerja sama antara Indonesia dan Rusia dalam bidang hilirisasi bauksit dan produksi pupuk.,Akun ini hanya menyampaikan informasi tentang rencana kerja sama Indonesia-Rusia tanpa menunjukkan si


=== Rekap stance terhadap Presiden Prabowo dan pemerintah per tema ===


stance_final,oposisi,moderat,supporter,total
theme,,,,
agenda_pertemuan,0,3,1,4
kerja_sama_bilateral,0,1,8,9
reaksi_geopolitik,3,2,4,9


# Step 7: Ringkasan per tema


In [ ]:
THEME_FIELDS = ["ringkasan", "narasi_supporter", "narasi_moderat", "narasi_oposisi"]
THEME_SCHEMA = {"type": "object", "properties": {f: {"type": "string"} for f in THEME_FIELDS}, "required": THEME_FIELDS}

THEME_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}
Komposisi sikap KOL di tema ini: {composition}

Ringkasan tiap KOL, urut dari yang paling berpengaruh:
{kols}

Tugas (HANYA berdasarkan ringkasan di atas, jangan menambah fakta dari luar):
1. ringkasan: 3-4 kalimat tentang apa yang dibicarakan para KOL di tema ini dan ke mana arah percakapannya.
2. narasi_supporter: 1-2 kalimat argumen utama kubu supporter. Tulis "-" kalau tidak ada supporter.
3. narasi_moderat: 1-2 kalimat argumen utama kubu moderat. Tulis "-" kalau tidak ada.
4. narasi_oposisi: 1-2 kalimat argumen utama kubu oposisi. Tulis "-" kalau tidak ada.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def summarize_theme(theme, kols):
    """Satu panggilan LLM lokal per tema, dari summary KOL (bukan cuitan mentah, supaya prompt tetap pendek)."""
    counts = kols["stance_final"].value_counts()
    content = THEME_PROMPT.format(
        topic=TOPIC, theme=theme.replace("_", " "), target=STANCE_TARGET,
        composition=", ".join(f"{s} {counts.get(s, 0)}" for s in STANCES),
        kols="\n".join(f"- [{r.stance_final}] {r.author_id}: {r.summary}" for r in kols.itertuples()),
    )
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object", "schema": THEME_SCHEMA},
        temperature=0.2, max_tokens=700,
    )
    out = json.loads(r["choices"][0]["message"]["content"])
    return {f: str(out.get(f) or "").strip() for f in THEME_FIELDS}


theme_records = []
usable = df_kol[(df_kol["summary"] != "") & df_kol["relevan"].ne(False)]   # KOL salah tema tidak ikut
for theme, kols in usable.sort_values("rank").groupby("theme"):
    try:
        out = summarize_theme(theme, kols)
    except Exception as e:
        print(f"[ERROR] tema {theme}: {type(e).__name__}: {e}")
        out = dict.fromkeys(THEME_FIELDS, "")
    counts = kols["stance_final"].value_counts()
    theme_records.append({"theme": theme, "jumlah_kol": len(kols),
                          **{s: int(counts.get(s, 0)) for s in STANCES},
                          "kol_teratas": ", ".join(kols["author_id"].head(3)), **out})

df_theme_summary = pd.DataFrame(theme_records)
print(f"[OK] Ringkasan {len(df_theme_summary)} tema selesai")
if LLM_TEST_N is not None:
    print("[TEST] Ringkasan tema ini baru dari KOL mode tes. Jalankan ulang setelah Step 6B dengan LLM_TEST_N = None.")
display(df_theme_summary)

[OK] Ringkasan 3 tema selesai


,theme,jumlah_kol,oposisi,moderat,supporter,kol_teratas,ringkasan,narasi_supporter,narasi_moderat,narasi_oposisi
0,agenda_pertemuan,4,0,3,1,"troublemakerid, @bambangpe, Akhand Bharat","Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin, dengan fokus pada agenda pertemuan dan gaya kerja Presiden Prabowo. Percakapan ini didominasi oleh pandangan positif dan netral, dengan sedikit kritik terhadap gaya kerja Presiden Prabowo.","Kubu supporter memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, menekankan pidato Prabowo tentang politik luar negeri bebas aktif dan komitmen program domestik Indonesia.","Kubu moderat membagikan artikel dan foto terkait pertemuan Presiden Prabowo dengan Putin, serta membandingkan gaya kerja Presiden Prabowo dengan Presiden Jokowi.",-
1,kerja_sama_bilateral,9,0,1,8,"@admkuznetsov45, @Pororo266, @latipiaa","Para KOL di media sosial Indonesia membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin dalam konteks kerja sama bilateral. Percakapan ini didominasi oleh dukungan terhadap upaya pemerintah Indonesia untuk meningkatkan ketahanan energi dan memperkuat kerja sama ekonomi dengan Rusia.","Kubu supporter memuji kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin sebagai langkah strategis untuk meningkatkan kerja sama bilateral dalam bidang energi, ekonomi, dan industri.",Kubu moderat membahas rencana kerja sama antara Indonesia dan Rusia dalam bidang hilirisasi bauksit dan produksi pupuk.,-
2,reaksi_geopolitik,9,3,2,4,"@LambeSahamjja, Meta AI, Dedi","Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin, serta reaksi geopolitik terkait isu alutsista dan potensi perang Eropa-Rusia. Percakapan ini juga membahas kebijakan luar negeri bebas aktif Indonesia dan posisi Indonesia di antara negara-negara besar.","Kubu supporter memuji keberanian dan ketegasan Presiden Prabowo dalam kebijakannya, terutama dalam hal ekonomi dan hubungan internasional, serta mendukung kunjungannya ke Rusia dan pembelian alutsista.","Kubu moderat membahas kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Vladimir Putin dalam konteks KTT BRICS, serta menekankan sikap tegas Prabowo mengenai posisi Indonesia di antara negara-negara besar.","Kubu oposisi mengkritik kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin karena dianggap tidak efektif dan hanya sebagai kunjungan formal tanpa hasil yang signifikan, serta mempertanyakan ketiadaan Presiden Prabowo dalam foto pertemuan Brics dan pidato Prabowo yang dikritik oleh pejabat Rusia, termasuk Putin."
